In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url=os.environ.get("GITHUB_BASE_URL"),
    api_key=os.environ.get("GITHUB_TOKEN")
)

contexto_red = """
INFORMACIÓN DE TARIFAS (General):
- Horario Punta (07:00-08:59 y 18:00-19:59): $840
- Horario Valle (09:00-17:59 y 20:00-20:44): $760
- Horario Bajo (06:00-06:59 y 20:45-23:00): $680
- Tarifa Estudiante (TNE) y Adulto Mayor: $240 en todo horario.

ESTRUCTURA DE LÍNEAS Y COMBINACIONES CLAVE:
- Línea 1 (Roja): Terminales San Pablo / Los Dominicos. Combina con L2, L3, L4 (Tobalaba), L5 y L6.
- Línea 4 (Azul): Terminales Tobalaba / Plaza Puente Alto. Combina con L1 (Tobalaba), L3 y L5.
"""

horario_actual = "Día hábil, 18:30 hrs (Horario Punta)"
alerta_operativa = "Línea 1 presenta retrasos por asistencia a pasajero en estación Baquedano."

prompt_sistema = f"""
Actúa como el asistente virtual oficial de servicio al cliente de Metro de Santiago.
Contexto de la red: {contexto_red}
Horario actual: {horario_actual}
Alertas operativas: {alerta_operativa}

Reglas:
1. Indica la línea inicial, combinaciones y estación final.
2. Informa la tarifa exacta del viaje.
3. Advierte sobre alertas operativas.
"""

# ¡AQUÍ ESTÁ LA MEMORIA! Una simple lista que guarda los mensajes
historial_chat = [
    {"role": "system", "content": prompt_sistema}
]

print("🚇 Asistente de Metro iniciado (Escribe 'salir' para terminar)\n")

while True:
    pregunta = input("Tú: ")
    if pregunta.lower() == 'salir':
        print("Asistente: ¡Buen viaje! Cerrando sistema.")
        break
        
    # 1. Guardamos tu pregunta en la memoria
    historial_chat.append({"role": "user", "content": pregunta})
    
    try:
        # 2. Le enviamos TODA la memoria al modelo
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=historial_chat,
            temperature=0.7, 
            max_tokens=600
        )
        
        respuesta_asistente = response.choices[0].message.content
        print(f"\nAsistente: {respuesta_asistente}\n")
        
        # 3. Guardamos la respuesta del asistente en la memoria para que no se le olvide
        historial_chat.append({"role": "assistant", "content": respuesta_asistente})

    except Exception as e:
        print(f"Error de conexión: {e}")
        break